#Part 4 — GenAI: Turn call-centre notes into structured reasons

##Task 4.1 — One validated call

In [ ]:
!pip install -q google-genai pydantic

In [1]:
import json
import pandas as pd
import time

from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

In [2]:
API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client created successfully.")

Paste your Gemini API key: d;la
Gemini client created successfully.


In [ ]:
MODEL_NAME = "gemini-3.8-flash"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [ ]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [ ]:
SYSTEM_TEXT = """
You are a healthcare appointment follow-up analysis assistant.

Your task is to analyse patient call-centre notes
and identify the most likely reason for a missed appointment.

Use only the information provided in the note.
Do not invent reasons or make unsupported assumptions.
Return the requested structured JSON response.
"""

In [ ]:
def build_craft_prompt(note_text):

    prompt = f"""
    CONTEXT:
    A healthcare clinic wants to understand why patients
    miss their scheduled appointments.

    ROLE:
    You are an expert healthcare call-centre data analyst.

    ACTION:
    Analyse the following patient follow-up note.
    Identify the main reason for the missed appointment.

    Allowed root_cause values:
    {ALLOWED_CAUSES}

    If the reason is unclear or unsupported by the note,
    select "other" rather than guessing.

    TONE:
    Be objective, concise, accurate and professional.

    FORMAT:
    Return valid JSON containing exactly these four fields:
    root_cause,
    confidence,
    summary,
    recommended_action.

    Confidence must be between 0 and 1.
    Summary must contain at most 25 words.
    Do not include additional text outside the JSON.

    PATIENT NOTE:
    {note_text}
    """

    return prompt

In [ ]:
API_CALL_COUNT = 0
LAST_RAW_RESPONSE = ""

def analyse_note(note_text):

    global API_CALL_COUNT
    global LAST_RAW_RESPONSE

    prompt = build_craft_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA))

    API_CALL_COUNT += 1

    LAST_RAW_RESPONSE = response.text

    data = json.loads(response.text)

    return data

In [ ]:
notes_df = pd.read_csv("followupnotes.csv")

note_N001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result_N001 = analyse_note(note_N001)

print("Analysis of note N001:")
print(result_N001)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 22h29m2.673638976s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

###Task 4.2 — Validate and process the 10 notes (7 marks)

In [ ]:
class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)

In [ ]:
def validate_root_cause(data):

    if data.root_cause not in ALLOWED_CAUSES:

        raise ValueError(
            f"Invalid root_cause: {data.root_cause}"
        )

    return True

In [ ]:
notes_df = pd.read_csv("followupnotes.csv")

first_10_notes = notes_df.head(10)

print(first_10_notes)

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...


In [ ]:
results = []

for index, row in first_10_notes.iterrows():

    note_text = row["note_text"]

    result = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check missing or blank notes first
    if pd.isna(note_text) or str(note_text).strip() == "":

        result["root_cause"] = "other"
        result["confidence"] = 0.0
        result["summary"] = "No note recorded"
        result["recommended_action"] = "Re-call patient"
        result["validation_status"] = "skipped_blank"

    else:

        try:

            # Call Gemini API
            data = analyse_note(str(note_text))

            # Preserve the original response
            result["raw_response"] = LAST_RAW_RESPONSE

            # Copy the fields returned by the model
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:

                if field in data:
                    result[field] = data[field]

            # Validate using Pydantic
            insight = FollowupInsight(**data)

            # Check allowed root cause
            validate_root_cause(insight)

            result["validation_status"] = "valid"

        except Exception as e:

            result["validation_status"] = "invalid"

            print(
                f"Validation error for {row['note_id']}: {e}"
            )

        # Pause between API calls
        time.sleep(13)

    results.append(result)


# Convert results into a DataFrame
analysed_df = pd.DataFrame(results)

print(analysed_df)

Validation error for N001: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 22h28m37.895697793s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See htt

In [ ]:
output_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

analysed_df = analysed_df[output_columns]

analysed_df.to_csv(
    "notes_analysed.csv",
    index=False
)

print("notes_analysed.csv saved successfully.")

notes_analysed.csv saved successfully.


In [ ]:
print(
    analysed_df["validation_status"].value_counts()
)

print("\nAnalysed notes:")
display(analysed_df)

validation_status
invalid          9
skipped_blank    1
Name: count, dtype: int64

Analysed notes:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,,,,,invalid,
1,N002,A0897,,,,,invalid,
2,N003,A0526,,,,,invalid,
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,,,,,invalid,
5,N006,A0507,,,,,invalid,
6,N007,A0333,,,,,invalid,
7,N008,A0222,,,,,invalid,
8,N009,A0977,,,,,invalid,
9,N010,A0596,,,,,invalid,


###Task 4.3 — Break it on purpose

In [ ]:
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot appointment",
    "recommended_action": "Send a reminder"
}

try:

    insight = FollowupInsight(**bad_reply_1)

    validate_root_cause(insight)

except ValidationError as e:

    print("Error for invalid confidence:")
    print(e)

Error for invalid confidence:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [ ]:
bad_reply_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.8,
    "summary": "Patient had transport difficulties"
}

try:

    insight = FollowupInsight(**bad_reply_2)

    validate_root_cause(insight)

except ValidationError as e:

    print("Error for missing recommended_action:")
    print(e)

Error for missing recommended_action:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...transport difficulties'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


Part (b): Compare V1 and V2 prompts
We will compare:
- V1: Simple prompt — "Classify this note".
- V2: CRAFT prompt — includes allowed values and tells the model to choose other rather than guess.

In [ ]:
def build_v1_prompt(note_text):

    return f"Classify this note:\n{note_text}"


def build_v2_prompt(note_text):

    return build_craft_prompt(note_text)

In [ ]:
def compare_prompt(note_text, prompt_version):

    global API_CALL_COUNT

    if prompt_version == "V1":

        prompt = build_v1_prompt(note_text)

    else:

        prompt = build_v2_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    API_CALL_COUNT += 1

    raw = response.text

    try:

        data = json.loads(raw)

        cause = data.get("root_cause", "")

        if cause in ALLOWED_CAUSES:

            status = "valid"

        else:

            status = "invalid"

    except Exception:

        data = {}

        cause = ""

        status = "invalid"

    return {
        "prompt": prompt_version,
        "root_cause": cause,
        "validation_status": status,
        "raw_response": raw
    }

In [ ]:
note_N007 = notes_df.loc[
    notes_df["note_id"] == "N007",
    "note_text"
].iloc[0]

note_N009 = notes_df.loc[
    notes_df["note_id"] == "N009",
    "note_text"
].iloc[0]

print("N007:", note_N007)
print("N009:", note_N009)

N007: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N009: Phone was switched off, could not reach the patient after three attempts.


In [ ]:
v1_N007 = compare_prompt(
    str(note_N007),
    "V1"
)

time.sleep(13)

v1_N009 = compare_prompt(
    str(note_N009),
    "V1"
)

time.sleep(13)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 22h7m36.68126201s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.go

In [ ]:
v2_N007 = analysed_df.loc[
    analysed_df["note_id"] == "N007"
].iloc[0]

v2_N009 = analysed_df.loc[
    analysed_df["note_id"] == "N009"
].iloc[0]

In [ ]:
comparison_results = [

    {
        "Note": "N007",
        "Prompt": "V1",
        "Root Cause": v1_N007["root_cause"],
        "Validation": v1_N007["validation_status"]
    },

    {
        "Note": "N007",
        "Prompt": "V2",
        "Root Cause": v2_N007["root_cause"],
        "Validation": v2_N007["validation_status"]
    },

    {
        "Note": "N009",
        "Prompt": "V1",
        "Root Cause": v1_N009["root_cause"],
        "Validation": v1_N009["validation_status"]
    },

    {
        "Note": "N009",
        "Prompt": "V2",
        "Root Cause": v2_N009["root_cause"],
        "Validation": v2_N009["validation_status"]
    }

]

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

In [ ]:
print(
    "Total Gemini API calls across Tasks 4.1–4.3:",
    API_CALL_COUNT
)

###Task 4.4 — Recommendation and GitHub

In [ ]:
appointments_df = pd.read_csv("appointments_clean.csv")

print(appointments_df.head())

  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  
1         

In [ ]:
merged_df = pd.merge(
    analysed_df,
    appointments_df,
    on="appointment_id",
    how="left"
)

print("Merged dataset:")
display(merged_df.head())

Merged dataset:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,,,,,invalid,,P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,,,,,invalid,,P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,,,,,invalid,,P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,,,,,invalid,,P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1


In [ ]:
cross_tab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print("Root cause vs Reminder sent:")

display(cross_tab)

Root cause vs Reminder sent:


reminder_sent,No,Yes
root_cause,,
,4,5
other,1,0


In [ ]:
cross_tab.to_csv("root_cause_reminder_crosstab.csv")

print("Cross-tabulation saved successfully.")

Cross-tabulation saved successfully.


In [ ]:
print("Total notes analysed:", len(analysed_df))

print(
    "Valid notes:",
    (analysed_df["validation_status"] == "valid").sum()
)

print(
    "Invalid notes:",
    (analysed_df["validation_status"] == "invalid").sum()
)

print(
    "Blank notes:",
    (analysed_df["validation_status"] == "skipped_blank").sum()
)

print("\nRoot cause distribution:")

print(analysed_df["root_cause"].value_counts())

print("\nReminder cross-tabulation:")

display(cross_tab)

Total notes analysed: 10
Valid notes: 0
Invalid notes: 9
Blank notes: 1

Root cause distribution:
root_cause
         9
other    1
Name: count, dtype: int64

Reminder cross-tabulation:


reminder_sent,No,Yes
root_cause,,
,4,5
other,1,0
